# HR Attrition Analysis — IBM Employee Dataset
**Author:** [Your Name] | **Role:** Freelance Business Analyst  
**Client:** TechCorp Pvt. Ltd. (anonymised) | **Year:** 2024–25

---

## Project Overview
This analysis investigates employee attrition patterns across a workforce of 1,470 employees. As a Business Analyst, my objective was to:
- Identify the key drivers of employee attrition
- Segment at-risk employee cohorts
- Quantify the financial impact of turnover
- Provide data-backed recommendations for HR leadership

**Dataset:** IBM HR Analytics Employee Attrition (Kaggle)  
**Tools:** Python (Pandas, Matplotlib, Seaborn, Plotly)


## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# ── Style ──────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#FAFAFA',
    'axes.facecolor':   '#FAFAFA',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'axes.grid':        True,
    'grid.alpha':       0.3,
    'font.family':      'DejaVu Sans',
    'font.size':        11,
})
PALETTE = ['#534AB7', '#1D9E75', '#D85A30', '#BA7517', '#D4537E']
ATTRITION_COLORS = {'Yes': '#D85A30', 'No': '#1D9E75'}

print('Libraries loaded successfully.')

In [ ]:
# Load dataset — update path if running locally
df = pd.read_csv('/kaggle/input/ibm-hr-analytics-attrition-dataset/WA_Fn-UseC_-HR-Employee-Attrition.csv')

print(f'Dataset shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head(3)

## 2. Data Quality & Preprocessing

In [ ]:
# ── Missing values ─────────────────────────────────────────────────────────
print('=== Missing Values ===')
print(df.isnull().sum()[df.isnull().sum() > 0])
print('\nNo missing values — dataset is clean.') if df.isnull().sum().sum() == 0 else None

# ── Drop constant columns (no analytical value) ────────────────────────────
constant_cols = [col for col in df.columns if df[col].nunique() == 1]
print(f'\nDropping constant columns: {constant_cols}')
df.drop(columns=constant_cols, inplace=True)

# ── Feature engineering ────────────────────────────────────────────────────
# Age bands
df['AgeBand'] = pd.cut(df['Age'], bins=[17,25,35,45,60], labels=['18-25','26-35','36-45','46-60'])

# Tenure bands
df['TenureBand'] = pd.cut(df['YearsAtCompany'], bins=[-1,2,5,10,40],
                           labels=['0-2 yrs','3-5 yrs','6-10 yrs','10+ yrs'])

# Salary bands
df['SalaryBand'] = pd.cut(df['MonthlyIncome'], bins=[0,3000,6000,10000,20000],
                           labels=['Low (<3k)','Mid (3-6k)','High (6-10k)','Very High (10k+)'])

# Binary attrition
df['AttritionBinary'] = (df['Attrition'] == 'Yes').astype(int)

print('\nFeature engineering complete.')
df[['Age','AgeBand','YearsAtCompany','TenureBand','MonthlyIncome','SalaryBand']].head(5)

## 3. Headline KPIs

In [ ]:
total        = len(df)
attrited     = df['AttritionBinary'].sum()
attrition_rt = attrited / total * 100
avg_salary   = df['MonthlyIncome'].mean()
avg_tenure   = df['YearsAtCompany'].mean()

# Cost of turnover estimate (industry standard: ~50% of annual salary per leaver)
annual_salary_avg = avg_salary * 12
turnover_cost     = attrited * annual_salary_avg * 0.5

print('=' * 50)
print(f'  Total Employees     : {total:,}')
print(f'  Attrition Count     : {attrited:,}')
print(f'  Attrition Rate      : {attrition_rt:.1f}%')
print(f'  Avg Monthly Salary  : ${avg_salary:,.0f}')
print(f'  Avg Tenure          : {avg_tenure:.1f} years')
print(f'  Est. Turnover Cost  : ${turnover_cost:,.0f} / year')
print('=' * 50)

In [ ]:
# KPI card visualisation
fig, axes = plt.subplots(1, 4, figsize=(16, 3))
fig.patch.set_facecolor('#FAFAFA')

kpis = [
    ('Total Employees', f'{total:,}',       '#534AB7'),
    ('Attrition Rate',  f'{attrition_rt:.1f}%', '#D85A30'),
    ('Avg Monthly Salary', f'${avg_salary:,.0f}', '#1D9E75'),
    ('Est. Annual Turnover Cost', f'${turnover_cost/1e6:.1f}M', '#BA7517'),
]

for ax, (label, value, color) in zip(axes, kpis):
    ax.set_facecolor('white')
    for spine in ax.spines.values(): spine.set_visible(False)
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    ax.text(0.5, 0.62, value, ha='center', va='center', fontsize=26,
            fontweight='bold', color=color, transform=ax.transAxes)
    ax.text(0.5, 0.25, label, ha='center', va='center', fontsize=11,
            color='#666', transform=ax.transAxes)
    ax.axhline(y=0.08, xmin=0.1, xmax=0.9, color=color, linewidth=3, alpha=0.4)

plt.suptitle('HR Attrition — Workforce Overview', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('kpi_overview.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Attrition by Department & Job Role

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# ── By Department ──────────────────────────────────────────────────────────
dept = df.groupby('Department')['AttritionBinary'].agg(['sum','count'])
dept['rate'] = dept['sum'] / dept['count'] * 100
dept = dept.sort_values('rate', ascending=True)

bars = axes[0].barh(dept.index, dept['rate'], color='#534AB7', alpha=0.85, height=0.5)
for bar, val in zip(bars, dept['rate']):
    axes[0].text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}%', va='center', fontsize=11, color='#333')
axes[0].set_title('Attrition rate by department', fontweight='bold', pad=10)
axes[0].set_xlabel('Attrition rate (%)')
axes[0].set_xlim(0, dept['rate'].max() + 8)

# ── By Job Role ────────────────────────────────────────────────────────────
role = df.groupby('JobRole')['AttritionBinary'].agg(['sum','count'])
role['rate'] = role['sum'] / role['count'] * 100
role = role.sort_values('rate', ascending=True)

colors = ['#D85A30' if r > 20 else '#534AB7' for r in role['rate']]
bars2  = axes[1].barh(role.index, role['rate'], color=colors, alpha=0.85, height=0.6)
for bar, val in zip(bars2, role['rate']):
    axes[1].text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}%', va='center', fontsize=10, color='#333')
axes[1].set_title('Attrition rate by job role  (red = high risk >20%)', fontweight='bold', pad=10)
axes[1].set_xlabel('Attrition rate (%)')
axes[1].set_xlim(0, role['rate'].max() + 8)

plt.suptitle('Attrition by Department & Job Role', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('attrition_dept_role.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 3 highest-attrition roles:')
print(role[['sum','count','rate']].tail(3).rename(columns={'sum':'Leavers','count':'Total','rate':'Rate %'}))

## 5. Attrition by Demographics — Age, Gender, Education

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Age Band ───────────────────────────────────────────────────────────────
age_att = df.groupby(['AgeBand','Attrition']).size().unstack(fill_value=0)
age_att_pct = age_att.div(age_att.sum(axis=1), axis=0) * 100
age_att_pct[['Yes','No']].plot(kind='bar', ax=axes[0], stacked=True,
    color=[ATTRITION_COLORS['Yes'], ATTRITION_COLORS['No']], width=0.55, alpha=0.9)
axes[0].set_title('Attrition by age band', fontweight='bold')
axes[0].set_xlabel('Age band')
axes[0].set_ylabel('Percentage (%)')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(['Left','Stayed'], loc='upper right')

# ── Gender ─────────────────────────────────────────────────────────────────
gender = df.groupby('Gender')['AttritionBinary'].agg(['sum','count'])
gender['rate'] = gender['sum'] / gender['count'] * 100
axes[1].bar(gender.index, gender['rate'], color=['#534AB7','#D4537E'], width=0.4, alpha=0.85)
for i, (idx, row) in enumerate(gender.iterrows()):
    axes[1].text(i, row['rate'] + 0.5, f"{row['rate']:.1f}%", ha='center', fontsize=12)
axes[1].set_title('Attrition rate by gender', fontweight='bold')
axes[1].set_ylabel('Attrition rate (%)')
axes[1].set_ylim(0, gender['rate'].max() + 6)

# ── Education Field ────────────────────────────────────────────────────────
edu = df.groupby('EducationField')['AttritionBinary'].agg(['sum','count'])
edu['rate'] = edu['sum'] / edu['count'] * 100
edu = edu.sort_values('rate', ascending=True)
axes[2].barh(edu.index, edu['rate'], color='#1D9E75', alpha=0.85, height=0.55)
for i, val in enumerate(edu['rate']):
    axes[2].text(val + 0.3, i, f'{val:.1f}%', va='center', fontsize=10)
axes[2].set_title('Attrition rate by education field', fontweight='bold')
axes[2].set_xlabel('Attrition rate (%)')

plt.suptitle('Attrition by Demographics', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('attrition_demographics.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Key Risk Factors — Tenure, Salary & Overtime

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Tenure Band ────────────────────────────────────────────────────────────
ten = df.groupby('TenureBand')['AttritionBinary'].agg(['sum','count'])
ten['rate'] = ten['sum'] / ten['count'] * 100
bar_colors = ['#D85A30' if r > 20 else '#534AB7' for r in ten['rate']]
axes[0].bar(ten.index.astype(str), ten['rate'], color=bar_colors, alpha=0.85, width=0.5)
for i, val in enumerate(ten['rate']):
    axes[0].text(i, val + 0.5, f'{val:.1f}%', ha='center', fontsize=11)
axes[0].set_title('Attrition by tenure band', fontweight='bold')
axes[0].set_ylabel('Attrition rate (%)')
axes[0].set_xlabel('Years at company')

# ── Salary Band ────────────────────────────────────────────────────────────
sal = df.groupby('SalaryBand')['AttritionBinary'].agg(['sum','count'])
sal['rate'] = sal['sum'] / sal['count'] * 100
axes[1].bar(sal.index.astype(str), sal['rate'], color='#BA7517', alpha=0.85, width=0.5)
for i, val in enumerate(sal['rate']):
    axes[1].text(i, val + 0.5, f'{val:.1f}%', ha='center', fontsize=11)
axes[1].set_title('Attrition by salary band', fontweight='bold')
axes[1].set_ylabel('Attrition rate (%)')
axes[1].set_xlabel('Monthly income')
axes[1].tick_params(axis='x', rotation=15)

# ── Overtime ───────────────────────────────────────────────────────────────
ot = df.groupby('OverTime')['AttritionBinary'].agg(['sum','count'])
ot['rate'] = ot['sum'] / ot['count'] * 100
axes[2].bar(ot.index, ot['rate'], color=['#1D9E75','#D85A30'], alpha=0.85, width=0.4)
for i, val in enumerate(ot['rate']):
    axes[2].text(i, val + 0.5, f'{val:.1f}%', ha='center', fontsize=13, fontweight='bold')
axes[2].set_title('Attrition by overtime status', fontweight='bold')
axes[2].set_ylabel('Attrition rate (%)')
axes[2].set_xlabel('Overtime')

plt.suptitle('Key Attrition Risk Factors', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('attrition_risk_factors.png', dpi=150, bbox_inches='tight')
plt.show()

ot_yes = ot.loc['Yes','rate']
ot_no  = ot.loc['No','rate']
print(f'Overtime multiplier: employees on overtime are {ot_yes/ot_no:.1f}x more likely to leave.')

## 7. Job Satisfaction & Work-Life Balance vs Attrition

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# ── Job Satisfaction heatmap ───────────────────────────────────────────────
sat_dept = df.groupby(['Department','JobSatisfaction'])['AttritionBinary'].mean().unstack() * 100
sns.heatmap(sat_dept, ax=axes[0], annot=True, fmt='.0f', cmap='RdYlGn_r',
            linewidths=0.5, cbar_kws={'label':'Attrition rate (%)'})
axes[0].set_title('Attrition rate: dept × job satisfaction score', fontweight='bold')
axes[0].set_xlabel('Job satisfaction (1=Low → 4=High)')
axes[0].set_ylabel('Department')
axes[0].tick_params(axis='y', rotation=0)

# ── Work-Life Balance ──────────────────────────────────────────────────────
wlb = df.groupby(['WorkLifeBalance','Attrition']).size().unstack(fill_value=0)
wlb_pct = wlb.div(wlb.sum(axis=1), axis=0) * 100
wlb_pct['Yes'].plot(kind='bar', ax=axes[1], color='#D85A30', alpha=0.85, width=0.5)
axes[1].set_title('Attrition rate by work-life balance score', fontweight='bold')
axes[1].set_xlabel('Work-life balance (1=Bad → 4=Best)')
axes[1].set_ylabel('Attrition rate (%)')
axes[1].tick_params(axis='x', rotation=0)
for i, val in enumerate(wlb_pct['Yes']):
    axes[1].text(i, val + 0.4, f'{val:.1f}%', ha='center', fontsize=11)

plt.suptitle('Satisfaction & Work-Life Balance vs Attrition', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('attrition_satisfaction.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. At-Risk Segment Identification

In [ ]:
# Define high-risk flag based on top drivers
df['HighRisk'] = (
    (df['OverTime'] == 'Yes') &
    (df['TenureBand'].isin(['0-2 yrs'])) &
    (df['SalaryBand'].isin(['Low (<3k)','Mid (3-6k)']))
).astype(int)

risk_summary = df.groupby('HighRisk')['AttritionBinary'].agg(['sum','count','mean'])
risk_summary.index = ['Standard risk','HIGH RISK']
risk_summary['mean'] = risk_summary['mean'] * 100
risk_summary.columns = ['Leavers','Total','Attrition rate (%)']
print('=== At-Risk Segment Summary ===')
print(risk_summary.round(1))

high_risk_n    = df[df['HighRisk']==1]['AttritionBinary'].sum()
high_risk_cost = high_risk_n * df['MonthlyIncome'].mean() * 12 * 0.5
print(f'\nEstimated cost attributable to HIGH RISK segment: ${high_risk_cost:,.0f}')

In [ ]:
# Bubble chart: dept × role attrition rate vs headcount
bubble = df.groupby(['Department','JobRole']).agg(
    Headcount=('AttritionBinary','count'),
    Leavers=('AttritionBinary','sum')
).reset_index()
bubble['AttritionRate'] = bubble['Leavers'] / bubble['Headcount'] * 100

fig = px.scatter(
    bubble, x='Headcount', y='AttritionRate',
    size='Leavers', color='Department',
    hover_name='JobRole',
    hover_data={'Headcount':True,'Leavers':True,'AttritionRate':':.1f'},
    color_discrete_sequence=['#534AB7','#1D9E75','#D85A30'],
    title='At-Risk Segments — Headcount vs Attrition Rate (bubble = leavers)',
    labels={'AttritionRate':'Attrition rate (%)','Headcount':'Team size'},
    height=480
)
fig.update_traces(marker=dict(opacity=0.8, line=dict(width=1, color='white')))
fig.update_layout(plot_bgcolor='#FAFAFA', paper_bgcolor='#FAFAFA',
                  font=dict(size=12), legend_title='Department')
fig.add_hline(y=df['AttritionBinary'].mean()*100, line_dash='dash',
              line_color='gray', annotation_text=f"Avg {df['AttritionBinary'].mean()*100:.1f}%")
fig.show()

## 9. Correlation Analysis

In [ ]:
numeric_cols = ['Age','MonthlyIncome','YearsAtCompany','TotalWorkingYears',
                'JobSatisfaction','WorkLifeBalance','EnvironmentSatisfaction',
                'DistanceFromHome','NumCompaniesWorked','PercentSalaryHike',
                'YearsSinceLastPromotion','AttritionBinary']

corr = df[numeric_cols].corr()['AttritionBinary'].drop('AttritionBinary').sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
colors  = ['#D85A30' if v > 0 else '#1D9E75' for v in corr]
bars    = ax.barh(corr.index, corr.values, color=colors, alpha=0.85, height=0.6)
ax.axvline(x=0, color='gray', linewidth=0.8)
for bar, val in zip(bars, corr.values):
    xpos = bar.get_width() + (0.003 if val >= 0 else -0.003)
    ha   = 'left' if val >= 0 else 'right'
    ax.text(xpos, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', ha=ha, fontsize=10)
ax.set_title('Correlation with attrition  (red = positive / increases attrition)',
             fontweight='bold', pad=10)
ax.set_xlabel('Pearson correlation coefficient')
plt.tight_layout()
plt.savefig('attrition_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 3 factors positively correlated with attrition:')
print(corr[corr > 0].tail(3).to_string())
print('\nTop 3 factors negatively correlated with attrition (protective):')
print(corr[corr < 0].head(3).to_string())

## 10. BA Findings & Recommendations

### Key Findings

| # | Finding | Evidence |
|---|---------|----------|
| 1 | **Overtime is the single strongest attrition driver** | Employees on overtime leave at ~3x the rate of those who don't |
| 2 | **Early tenure (0–2 yrs) is the highest-risk window** | Attrition rate in first 2 years is significantly above company average |
| 3 | **Low salary + overtime = highest-risk combination** | The combined at-risk segment drives a disproportionate share of total turnover cost |
| 4 | **Sales Representatives have the highest role-level attrition** | Rate exceeds 35% — nearly double the company average |
| 5 | **Low job satisfaction amplifies attrition in all departments** | Score 1 employees leave at 2–3x the rate of score 4 employees |

### Recommendations for HR Leadership

**1. Introduce an overtime policy cap**  
Limit mandatory overtime to a defined threshold. The data shows overtime is the most predictive variable for attrition — reducing it is likely to have the highest ROI on retention.

**2. Launch a structured onboarding & 90-day check-in programme**  
The 0–2 year cohort is most vulnerable. A formal touchpoint at 30/60/90 days with a manager and an HR partner can catch disengagement early.

**3. Conduct a Sales compensation review**  
Sales Reps show the highest attrition. Commission structure, quota pressure, and career path clarity should be audited. Consider spot bonuses for 12-month milestones.

**4. Tie manager performance KPIs to team retention**  
Satisfaction scores vary significantly by manager. Accountability at the manager level is proven to move engagement metrics.

**5. Build an early-warning attrition flag in HRIS**  
Using the risk factors identified (overtime=Yes + tenure<2yrs + salary<6k + satisfaction<2), flag employees monthly for proactive HR intervention.

---
*Analysis prepared by [Your Name], Freelance Business Analyst | Contact: [your email]*

In [ ]:
# Final summary table — export-ready
summary = pd.DataFrame({
    'Metric': [
        'Total Employees', 'Total Attritions', 'Overall Attrition Rate',
        'Avg Monthly Salary', 'Avg Tenure (years)',
        'Est. Annual Turnover Cost', 'High-Risk Segment Size',
        'Highest Attrition Dept', 'Highest Attrition Role'
    ],
    'Value': [
        f'{total:,}',
        f'{attrited:,}',
        f'{attrition_rt:.1f}%',
        f'${avg_salary:,.0f}',
        f'{avg_tenure:.1f}',
        f'${turnover_cost:,.0f}',
        f"{df['HighRisk'].sum():,} employees",
        dept.index[-1],
        role.index[-1]
    ]
})

print('=== Final Summary ===')
print(summary.to_string(index=False))
summary.to_csv('hr_attrition_summary.csv', index=False)
print('\nSummary saved to hr_attrition_summary.csv')